In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src.agents import AgentDriverConfig, AgentDriver
from src.agents.configs import DEFAULT_AGENT_CONFIGS

from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig, QueryPreprocessor
from src.pipelines.qa.query_preprocessing.decomposition import QueryDecomposerConfig
from src.pipelines.qa.query_preprocessing.denoising import QueryDenoiserConfig
from src.pipelines.qa.query_preprocessing.enhancing import QueryEnhancerConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Пример работы с QueryPreprocess-стадией QA-пайплайна

In [4]:
adriver_config = AgentDriverConfig(name='ollama', agent_config=DEFAULT_AGENT_CONFIGS['ollama'])
pprint(adriver_config, width=200)
agent = AgentDriver.connect(adriver_config)

AgentDriverConfig(name='ollama',
                  agent_config=AgentConnectorConfig(gen_strategy={'num_predict': 2048, 'seed': 42, 'temperature': 0.0, 'top_k': 1},
                                                    credentials={'host': 'localhost', 'model': 'llama3.1:8b', 'port': 11437},
                                                    ext_params={'keep_alive': -1, 'timeout': 560, 'trials': 5}))


2. Инициализация QueryPreprocessor-стадии

In [5]:
qp_config = QueryPreprocessorConfig(
    denoising_config=QueryDenoiserConfig(lang='en'),
    enhancing_config=QueryEnhancerConfig(lang='en'),
    decomposition_config=QueryDecomposerConfig(lang='en')
)
pprint(qp_config, width=200, depth=1)

QueryPreprocessorConfig(lang='auto',
                        log=<src.utils.logger.Logger object at 0x7f3099878460>,
                        verbose=False,
                        denoising_config=QueryDenoiserConfig(lang='en',
                                                             log=<src.utils.logger.Logger object at 0x7f3099a45960>,
                                                             verbose=False,
                                                             agent_gen_stategy=None,
                                                             agent_tasks_config=QueryDenoiserAgentTasksConfig(task_to_selector_mapping={...},
                                                                                                              swremoval='v2',
                                                                                                              grammarcheck='v2'),
                                                             cache_table_name='qp_denoising_stag

In [6]:
q_prep = QueryPreprocessor(agent, qp_config)

3. Пример вызова QueryPreprocessor-методов

In [7]:
QUERY_EXAMPLES = ["When was Alexander Pushkin born?", "When were Pushkin and Dostoevsky born?"]

In [8]:
for query in QUERY_EXAMPLES:
    print("\nquery: ", query)
    transformed_query, rinfo, trace = q_prep.perform(query)
    print("return info: ",rinfo)
    print("transformed query:")
    pprint(transformed_query)



query:  When was Alexander Pushkin born?
return info:  ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
transformed query:
QueryPreprocessingInfo(base_query='When was Alexander Pushkin born?',
                       denoised_query='When was Alexander Pushkin born?',
                       enchanced_query='What is the date of birth of Alexander '
                                       'Pushkin?',
                       decomposed_query=['What is the name of the famous '
                                         'Russian poet?',
                                         'On which date was this person born?'],
                       processed_query=['What is the name of the famous '
                                        'Russian poet?',
                                        'On which date was this person born?'])

query:  When were Pushkin and Dostoevsky born?
return info:  ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
transf

In [12]:
trace.detailed_result.get_results_sequence()

[(<ModuleType.stage: 'stage'>,
  'denoiser',
  CompositeModuleSummaryResult(context={'positional_arguments': (QueryPreprocessingInfo(base_query='When were Pushkin and Dostoevsky born?', denoised_query=None, enchanced_query=None, decomposed_query=None, processed_query=None),)}, result='What are the birthdates of Pushkin and Dostoevsky?', status=<ReturnStatus.success: 0>, elapsed_time=2.09743)),
 (<ModuleType.stage: 'stage'>,
  'enhancer',
  CompositeModuleSummaryResult(context={'positional_arguments': (QueryPreprocessingInfo(base_query='When were Pushkin and Dostoevsky born?', denoised_query='What are the birthdates of Pushkin and Dostoevsky?', enchanced_query=None, decomposed_query=None, processed_query=None),)}, result='What are the birth dates of Aleksandr Pushkin and Fyodor Dostoevsky? \n\n(I changed "birthdates" to "birth dates", adding a plural form for consistency with the rest of the query.)', status=<ReturnStatus.success: 0>, elapsed_time=4.49214)),
 (<ModuleType.stage: 'stage'